## Workflow Walkthrough for Using SessionGroups to Automate Extraction of Multiple Sessions from a Parent Directory Containing Experimental Data

## Outline

**0. Setup + Config.** Initialise the inputs the later sections will consume.
  - 0.1. Imports.
  - 0.2. Set the data and YAML paths.
  - 0.3. Initialise `sessions` via `select_sessions(...)`. This produces the `{name: {path, ...}}` dict that section 2 uses. The knobs are opened up in section 2.1.
  - 0.4. Initialise `catalog` via `default_harp_catalog(...)`. A one-line default that section 1 explains and lets you customise.

**1. Creating Data Structure Extraction Catalogues using `DataStructureCatalog`.**
  - 1.1. What a `DataStructureSpec` is and how to write one.
  - 1.2. What a catalog is and how to edit it (add, remove, enable, disable).
  - 1.3. Using a catalog to load and align ONE session via the alignment pipeline (`load_session`, then `normalise_to_zero`, then `build_global_clock` and `build_index_tables`).

**2. Extracting Data Structures from Multiple Sessions.**
  - 2.1. Selecting sessions with `select_sessions`:
    a. Selector depth.
    b. Include / exclude lists.
    c. Extractors (labelling).
    d. Level names.
    e. Level selectors.
  - 2.2. Combining across sessions with `combine_sessions`.

**3. Querying and Interacting with the Combined Data.**

**4. Cross-Clock Data (Neuropixels, DLC). Sketch only.**


## 0. Setup + Config

This section initialises the two artefacts the rest of the notebook consumes: a `sessions` dict (the output of `select_sessions`) and a `catalog` (a `DataStructureCatalog`). Both are introduced here with the minimum needed to make a runnable example. The detailed behaviour of each is explained in section 1 (for the catalog) and section 2 (for the sessions selection).

### 0.1. Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from data_conduit.sessiongroups import (
    # Section 0.3 / 2.1: choosing sessions.
    select_sessions,
    DateTimeExtractor,
    NameExtractor,
    # Section 0.4 / 1: declaring what to extract.
    default_harp_catalog,
    DataStructureCatalog,
    DataStructureSpec,
    # Section 1.3: per-session alignment pipeline.
    load_session,
    normalise_to_zero,
    build_global_clock,
    build_index_tables,
    # Section 2.2 / 3: multi-session build + combine + tailor.
    build_sessions,
    combine_sessions,
    add_label_column,
    drop_columns,
)
from data_conduit.datasources.monosource import RotationData
from data_conduit.utils import starts_with
import itables

if itables is not None:
    itables.init_notebook_mode()
    itables.DEFAULT_MAX_ROWS = 100
    itables
warnings.filterwarnings('ignore')  # quieten the preset loaders' verbose hints
from IPython.display import display


### 0.2. Set the data and YAML paths

`DATA_DIR` is the folder containing your session folders (possibly nested, see section 2.1a for depth). `DEVICE_YAML` and `SOUNDCARD_YAML` are the HARP device definitions used by the preset loaders. Replace the values below with absolute paths to your own data. The defaults point at the example dataset shipped with this repo so the notebook runs as-is.

In [ ]:
def _example_root():
    '''Walk up from CWD to the folder containing pyproject.toml.

    Used only to locate the bundled example data. For your own data,
    set DATA_DIR to an absolute path and delete this helper.
    '''
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / 'pyproject.toml').exists():
            return parent
    return Path.cwd()


_root = _example_root()

# ---- EDIT THESE to point at your own data (absolute paths are simplest) ----
DATA_DIR       = _root / 'Q_C_Analysis_Workflow' / 'data' / 'FbR_M01569522' / 'Testing'
DEVICE_YAML    = _root / 'device.yml'
SOUNDCARD_YAML = _root / 'soundcard.yml'
# ----------------------------------------------------------------------------

assert DATA_DIR.exists(), f'DATA_DIR does not exist (got: {DATA_DIR})'
print('data folder    :', DATA_DIR)
print('device yaml    :', DEVICE_YAML)
print('soundcard yaml :', SOUNDCARD_YAML)

### 0.3. Initialise `sessions`

`select_sessions` walks `DATA_DIR` and returns a dict describing every session folder it finds. The shape of that dict is:

    {
        session_name: {
            'path': Path('.../<session_folder>'),
            <intermediate level metadata>,    # e.g. 'day': 'Day 1 - Tactile cue'
            <labels from extractors>,         # default: 'label' = session folder name
        },
        ...
    }

Every multi-session function in section 2 (`build_sessions`, `combine_sessions`) takes this dict (or a filtered subset of it) as its first argument. It is the entry point for the multi-session workflow.

The call below shows every argument explicitly at its default value:

- `root`: the directory to walk.
- `depth`: how many levels below `root` the session folders are. `depth=0` means session folders are the immediate children of `root`; `depth=1` means descend one level first (here `Day N / <session>`).
- `include`: optional list of session folder names to keep (whitelist).
- `exclude_names`: optional list to drop (blacklist). Mutually exclusive with `include`.
- `extractors`: how to label each session. `None` falls back to a single `NameExtractor` (label = the folder name as a string).
- `level_names`: friendly names for the intermediate levels. We call the one intermediate level `'day'` here.
- `**level_selectors`: optional `l{n}_selector` filters for the intermediate levels (e.g. `l0_selector='Day 1'`).

Each is opened up further in section 2.1.

In [ ]:
sessions = select_sessions(
    root           = DATA_DIR,
    depth          = 1,                       # sessions sit one level below DATA_DIR (Day N / <session>)
    include        = None,                    # None = take every session folder
    exclude_names  = None,                    # None = exclude nothing
    extractors     = None,                    # None = NameExtractor (label = folder name string)
    level_names    = ('day',),                # name the one intermediate level so it lands in metadata
    # **level_selectors                       # no per-level filters at this stage; see 2.1e
)

print(f'found {len(sessions)} session folders')
# Preview the first three entries to show the shape.
for name in list(sessions)[:3]:
    print(f'  {name}: {sessions[name]}')

### 0.4. Initialise `catalog`

A `DataStructureCatalog` is an ordered, editable list of `DataStructureSpec` records. Each spec describes one data structure to load from a session: its name, the reader function that loads it, whether it is currently enabled, whether it is required (load fails if missing) or optional (skipped if missing), and whether it lives on a cross-clock timebase needing TTL conversion (see section 4).

`default_harp_catalog` returns a catalog pre-filled with the lab's common HARP / Bonsai structures:

- `events`: ExperimentEvents CSV. Required, since it provides the reference clock everything else is aligned to.
- `nosepoke`: the 6-board / 18-port Nosepoke MultiDevice.
- `soundcard`: SoundCard registers.
- `camera`: Camera0Frames frame counts.
- `video`: VideoData CSV metadata. Optional.
- `session_settings`: SessionSettings. Optional.

Section 1 explains `DataStructureSpec` and `DataStructureCatalog` in depth and shows how to edit the catalog (add, remove, enable, disable). Section 1.3 then uses the catalog to load and align one session via the alignment pipeline.

In [ ]:
catalog = default_harp_catalog(
    device_yaml              = DEVICE_YAML,
    soundcard_yaml           = SOUNDCARD_YAML,
    include_video            = True,          # include the optional VideoData spec
    include_session_settings = True,          # include the optional SessionSettings spec
)

print('catalog contents:')
for spec in catalog:
    flags = []
    if spec.required:
        flags.append('required')
    if not spec.enabled:
        flags.append('disabled')
    tag = f'  ({", ".join(flags)})' if flags else ''
    print(f'  {spec.name:18}{tag}')

## 1. Creating Data Structure Extraction Catalogues using `DataStructureCatalog`

A `DataStructureCatalog` is the configuration object that tells the alignment pipeline (section 1.3) which data structures to read from each session and how to read them. Nothing in this module reads data; it only describes what to read. The actual loading is done by the alignment functions, which walk the catalog.

The catalog is deliberately mutable: it is a configuration object you build and then tweak in place (`add` / `remove` / `enable` / `disable`) before handing it to the loader.

### 1.1. What a `DataStructureSpec` is, and how to write one

A `DataStructureSpec` is a small record describing ONE data structure to load. It answers four questions about that structure:

1. What is it called (`name`)?
2. How do I load it (`reader`)?
3. Should I load it right now (`enabled`)?
4. What happens if it is missing (`required`)?

There is also a fifth field, `sync`, used only for cross-clock data (Neuropixels, DLC). It defaults to `None` for the common HARP / Bonsai case. See section 4.

The fields in detail:

- `name` (str): short identifier (e.g. `'nosepoke'`, `'events'`). Also becomes the PREFIX of the structure's members once loaded, so a Nosepoke's `'Activations'` array appears in the bundle as `'nosepoke:Activations'`. Names must be unique within a catalog.
- `reader` (callable): a one-argument function `path -> object` that loads the structure from one session directory. The returned object can be a source instance with `.data_arrays` (e.g. `SoundCard`, `Nosepoke`), a source instance with `.df` (e.g. `ExperimentEvents`), a bare DataArray / Dataset / DataFrame, or a plain dict mapping names to those. The loader knows how to coerce each of these into bundle members.
- `enabled` (bool, default `True`): whether the loader should currently process this spec. Setting it to `False` is how you temporarily leave a structure out without removing it from the catalog.
- `required` (bool, default `False`): what to do when this structure cannot be loaded. `True` raises a hard error; `False` (default) quietly skips it with a warning. The default is 'load it only if it is present', which is what you want for optional modalities.
- `sync` (dict | None, default `None`): cross-clock configuration. See section 4.

In practice a spec is one line: a tiny lambda wrapping a preset class. The example below builds one spec by hand for an `InnerRotation` reader, which is not in the default catalog.

In [ ]:
# A DataStructureSpec is just a dataclass record. Build one by hand:
inner_rotation_spec = DataStructureSpec(
    name     = 'inner_rotation',
    reader   = lambda p: RotationData(experiment_directory_path=p, device_type='InnerRotation').df,
    enabled  = True,                                # already the default
    required = False,                               # already the default
    sync     = None,                                # already the default
)

print('name    :', inner_rotation_spec.name)
print('reader  :', inner_rotation_spec.reader)     # the bound lambda
print('enabled :', inner_rotation_spec.enabled)
print('required:', inner_rotation_spec.required)
print('sync    :', inner_rotation_spec.sync)

### 1.2. What a catalog is, and how to edit it

A `DataStructureCatalog` is an ordered, editable collection of `DataStructureSpec` objects. Think of it as the shopping list of data structures the loader should look for, IN ORDER. Two things follow from that ordering:

1. Spec names must be unique within a catalog.
2. The loader processes specs in the order they were added (and so does `combine_sessions` later, when reconciling member names).

Unlike most objects in the sessiongroups package (which are immutable and return copies), a catalog is deliberately mutable. The four editing methods all return the catalog itself so calls can be chained:

- `catalog.add(spec)`: append a spec. Raises if a spec with that name already exists.
- `catalog.remove(name)`: drop a spec by name. No-op if the name is absent.
- `catalog.enable(name)` / `catalog.disable(name)`: flip the spec's `enabled` flag without removing it.

Two read-only views:

- `catalog.enabled_specs()`: the specs the loader will actually process, in catalog order. This is the method the alignment pipeline calls.
- `catalog.names`: every spec name (enabled or not), in order.

The catalog also behaves like a normal collection: `len(catalog)`, `for spec in catalog`, `'nosepoke' in catalog`, `catalog['nosepoke']`.

Below we use the `catalog` initialised in section 0.4 and walk through the editing API.

In [ ]:
# Start from the catalog initialised in 0.4. Print its starting state.
print('initial   :', catalog)
print('len       :', len(catalog))
print('names     :', catalog.names)

# Add the InnerRotation spec built in 1.1.
catalog.add(inner_rotation_spec)
print('after add :', catalog)

# Temporarily switch off video; the spec stays in the catalog but the
# loader will skip it on the next read.
catalog.disable('video')
print('after disable("video"):', catalog)

# Re-enable video so the rest of the notebook sees the full set.
catalog.enable('video')
print('after enable("video") :', catalog)

# Membership tests and direct access.
print("'nosepoke' in catalog  :", 'nosepoke' in catalog)
print("catalog['events']      :", catalog['events'])

# enabled_specs() is what the loader actually consumes.
print()
print('what the loader will process now:')
for spec in catalog.enabled_specs():
    print(f'  - {spec.name}')

### 1.3. Using a catalog to load and align ONE session

With a catalog in hand, the per-session alignment pipeline is a chain of small functions. Each step is opt-in and returns a new `Session` (or a sidecar). The four steps:

1. `load_session(session_path, catalog)`: walks the catalog, loads every enabled same-clock spec into a `Session` whose data is on each member's original timebase. No time math is done at this step.

2. `normalise_to_zero(session)`: returns a new `Session` with every time-bearing member shifted so the chosen reference starts at 0. The chosen `t0` is recorded in `session.metadata['t0']` for provenance. Defaults to `t0_from='session'`, which picks the earliest log across all time-bearing members (so nothing ends up negative).

3. `build_global_clock(session, timestep=...)`: returns a regular 1D `np.ndarray` of evenly-spaced time ticks spanning the reference stream. This is the shared ruler that the next step indexes against.

4. `build_index_tables(session, clock)`: returns a `dict[member_name, DataFrame]` mapping each global-clock tick to the matching sample index of every time-bearing member. This is what lets you ask 'at global tick N, which row of stream X applies?' without resampling.

Steps 3 and 4 are optional. If you only want each member shifted to t=0 without a shared clock, run steps 1 and 2 and stop. If you want raw timestamps, run only step 1.

The cell below runs all four steps on the first session in `sessions`, using the `catalog` from section 1.2. We stash the clock and index tables into `session.metadata` for inspection later in the notebook.

In [ ]:
# Pick the first session in the dict from 0.3.
one_path = next(iter(sessions.values()))['path']
print('aligning session:', one_path.name)
print()

# Step 1: load every enabled spec on the session's own timebase.
session = load_session(
    session_path        = one_path,
    data_structures     = catalog,
    prefix_data_arrays  = True,                 # default; prefixes members with spec name
    verbose             = False,                # set True to see one line per loaded/skipped spec
)
print('after load_session       :', session.names)

# Step 2: shift everything so the earliest log sits at t=0.
session = normalise_to_zero(
    session,
    time_coord = 'Time',                       # default
    t0_from    = 'session',                    # default; earliest time across all members
)
print('t0 (subtracted)          :', round(session.metadata['t0'], 3), 's')

# Step 3: build the shared global clock (1-second ticks here).
clock = build_global_clock(
    session,
    timestep         = 1.0,
    time_coord       = 'Time',                  # default
    start            = None,                    # None -> 0.0 (matches normalised session)
    end              = None,                    # None -> latest reference-stream time
    reference_stream = 'events',                # default; the master clock
)
print('global clock             :', len(clock), 'ticks')

# Step 4: per-member index tables onto the global clock.
index_tables = build_index_tables(
    session,
    clock,
    time_coord = 'Time',                       # default
    match_type = 'nearest',                    # default; alternatives: 'before', 'after'
)
print('index tables             :', list(index_tables))

# Stash the clock + tables onto metadata so the rest of the notebook can read them.
session.metadata['global_clock'] = clock
session.metadata['index_tables'] = index_tables

A look at what came back. The `events` table is a `DataFrame` whose `Time` index now starts at ~0 after normalisation. An index table maps each global-clock tick to a sample index in one bundle member.

In [ ]:
session['events'].head(6)

In [ ]:
session.metadata['index_tables']['nosepoke:Activations'].head(6)

## 2. Extracting Data Structures from Multiple Sessions

In section 0.3 we initialised `sessions` with `select_sessions` using default arguments. Here we open those arguments up. Each of `select_sessions`'s knobs is explained below with a worked example. The point is to see what each one does on its own; in real use you would compose them.

Section 2.2 then takes the resulting `sessions` dict and the catalog from section 1, runs the alignment pipeline (section 1.3) on each session, and stacks one chosen structure across all of them via `combine_sessions`.

### 2.1. Selecting sessions with `select_sessions`

#### a. Selector depth (`depth`)

`depth` is how many directory levels below `root` the session folders sit. Concretely:

- `depth=0` means the immediate subfolders of `root` are sessions.
- `depth=1` means descend one level first, and the subfolders at that next level are sessions. This is the case here: `DATA_DIR` is a `Testing` folder, which contains `Day N` subfolders, which contain the session folders.
- `depth=3` would mean descend three levels (e.g. mouse / phase / day), and the subfolders at the fourth level are sessions. This is the Q_C layout when you point at the top-level `data/` folder.

Changing `depth` does NOT just re-key the output: it changes which folders the walker treats as 'sessions' at all. The cell below shows what each depth yields on the same `DATA_DIR`.

In [ ]:
# depth=0: treat the immediate subfolders of DATA_DIR as sessions.
# Here that gives us the 'Day N' folders, not the actual session folders.
d0 = select_sessions(DATA_DIR, depth=0)
print(f'depth=0 ->  {len(d0)} entries (the Day folders themselves, not real sessions):')
for name in list(d0)[:3]:
    print(f'             {name}')

print()

# depth=1: descend one level first. Now we get the real session folders.
# We also name the intermediate level so it lands in metadata.
d1 = select_sessions(DATA_DIR, depth=1, level_names=('day',))
print(f'depth=1 ->  {len(d1)} entries (the real session folders):')
for name in list(d1)[:3]:
    print(f'             {name}: day={d1[name]["day"]}')

print()

# depth=2: descend two levels. The session folders themselves have no
# subfolders that look like sessions, so the walk returns nothing useful.
d2 = select_sessions(DATA_DIR, depth=2)
print(f'depth=2 ->  {len(d2)} entries (too deep: nothing to find)')

#### b. Include / exclude lists (`include`, `exclude_names`)

Once `depth` has fixed which folders count as sessions, `include` and `exclude_names` filter that set:

- `include=[name1, name2, ...]`: keep ONLY sessions whose folder name is in this list (whitelist).
- `exclude_names=[name1, ...]`: keep every session EXCEPT those named here (blacklist).
- With neither, every discovered session is kept.
- `include` and `exclude_names` are mutually exclusive; passing both raises.

The two arguments operate at the SESSION level only. To filter the intermediate levels (e.g. only descend the `Day 1` folder), use level selectors in 2.1e.

In [ ]:
all_names = list(sessions)
first_five = all_names[:5]

print('ALL     :', len(select_sessions(DATA_DIR, depth=1)), 'sessions (no filter)')
print('INCLUDE :', len(select_sessions(DATA_DIR, depth=1, include=first_five)),
      'sessions (only the 5 named)')
print('EXCLUDE :', len(select_sessions(DATA_DIR, depth=1, exclude_names=first_five)),
      'sessions (all except those 5)')
print()
print('first 5 (used as the include/exclude list):')
for n in first_five:
    print(f'  {n}')

#### c. Extractors (`extractors`)

An EXTRACTOR derives a label from each session folder and writes it into the session's metadata entry under its own key. By default `select_sessions` attaches a single `NameExtractor`, whose label is the folder name as a string under the key `'label'`. Pass `extractors=DateTimeExtractor()` to parse the folder name into a real `datetime` instead (key `'datetime'`), or a list of extractors to attach several labels at once.

Subclass `LabelExtractor` to add custom labels (see `extractors.py`).

In [ ]:
# 1) Default: NameExtractor only. Each entry carries 'label' as a string.
default = select_sessions(DATA_DIR, depth=1)
name = next(iter(default))
print('default                   :', default[name])

# 2) DateTimeExtractor: parses the folder name into a datetime under 'datetime'.
parsed = select_sessions(DATA_DIR, depth=1, extractors=DateTimeExtractor())
print('DateTimeExtractor         :', parsed[name])

# 3) Both: pass a list. Each extractor writes under its own key, so both land.
both = select_sessions(
    DATA_DIR, depth=1,
    extractors=[NameExtractor(), DateTimeExtractor()],
)
print('NameExtractor + DateTime  :', both[name])

#### d. Level names (`level_names`)

Every intermediate folder walked through on the way to a session lands in that session's metadata entry. `level_names` lets you give those intermediate levels readable keys (e.g. `'mouseID'`, `'phase'`, `'day'`) instead of the generic fallback (`'level_0'`, `'level_1'`, ...).

It is the same idea as naming the columns of a path tuple. If `level_names` is shorter than `depth`, the unnamed positions still fall back to `level_N`, so the walk never fails on a length mismatch.

In [ ]:
# Without level_names: the one intermediate level lands under 'level_0'.
no_names = select_sessions(DATA_DIR, depth=1)
name = next(iter(no_names))
print('without level_names  :', no_names[name])

# With level_names: the same intermediate level lands under 'day'.
with_names = select_sessions(DATA_DIR, depth=1, level_names=('day',))
print('with  level_names    :', with_names[name])

#### e. Level selectors (`**level_selectors`)

Level selectors filter the INTERMEDIATE walk at a chosen depth. They are passed as keyword arguments of the form `l{n}_selector`, where `n` is the depth being filtered (0-indexed below `root`). The semantics match data-conduit's `collect_dfs`:

- missing / `None`: wildcard. Keep every folder at that depth.
- exact string: keep only folders whose name equals the string.
- list of strings: keep only folders whose name is in the list.
- callable: keep folders for which `callable(name)` returns `True`. Use the helpers `starts_with`, `ends_with`, `contains` from `data_conduit.utils`, or write your own.

Level selectors only act on intermediate levels. The session level itself is filtered by `include` and `exclude_names` (see 2.1b).

In [ ]:
# Exact-string selector at depth 0 (the 'Day N' level here): only Day 1.
day1_only = select_sessions(
    DATA_DIR, depth=1, level_names=('day',),
    l0_selector='Day 1 - Tactile cue',
)
print(f'l0_selector="Day 1 - Tactile cue"  ->  {len(day1_only)} sessions')
for n in day1_only:
    print(f'  {n}: day={day1_only[n]["day"]}')

print()

# Callable selector at depth 0: any folder starting with 'Day 1'.
# This pulls in 'Day 1 - Tactile cue', 'Day 10 - ...', 'Day 11 - ...', 'Day 12 - ...' etc.
day1_prefix = select_sessions(
    DATA_DIR, depth=1, level_names=('day',),
    l0_selector=starts_with('Day 1'),
)
print(f'l0_selector=starts_with("Day 1") ->  {len(day1_prefix)} sessions across these days:')
for day in sorted({entry['day'] for entry in day1_prefix.values()}):
    print(f'  {day}')

### 2.2. Combining across sessions with `build_sessions` and `combine_sessions`

Two functions, one workflow:

1. `build_sessions(sessions_dict, catalog, ...)` runs the alignment pipeline (load + optionally normalise + optionally attach cross-clock + optionally build clock and index tables) on every session in the dict, attaches the selection metadata to each, and returns a `SessionGroup` ordered by `sort_by` (default `('label',)`). It is the multi-session counterpart of section 1.3.

2. `combine_sessions(group, type_name='...')` stacks one chosen structure (e.g. `'nosepoke:Activations'`) across the group along `dim` (default `'Time'`), tagging every row or sample with the source session's label. The result keeps its NATIVE type: a Nosepoke stays an `xr.DataArray` queryable via `.ulookup`, a trial table stays a `pd.DataFrame`. If you omit `type_name`, every member is combined and a single combined `Session` is returned.

`build_sessions` exposes the same alignment knobs we walked through in section 1.3, packaged as keyword arguments: `normalise`, `t0_from`, `time_coord`, `attach_cross_clock_specs`, `ttl_sync`, `global_clock` (a dict carrying `timestep`, `start`, `end`, etc.). Sidecars (the fitted TTL models, the global clock, the per-member index tables) are stashed onto each session's metadata for later inspection.

The cell below runs the multi-session build on all ~29 sessions in `DATA_DIR`, so it does the real work and may take a moment.

In [ ]:
group = build_sessions(
    selection                = sessions,
    data_structures          = catalog,
    label                    = 'Testing folder (all sessions)',   # group-level label
    sort_by                  = ('label',),                        # order by session folder name (default)
    normalise                = False,                              # run normalise_to_zero on each session
    t0_from                  = 'session',                         # default; earliest log across members
    time_coord               = 'Time',                            # default
    attach_cross_clock_specs = True,                              # default; no-op without sync specs in catalog
    ttl_sync                 = None,                              # no cross-clock defaults
    global_clock             = {'timestep': 1.0},                 # build per-session clock + index tables
    prefix_data_arrays       = True,                              # default
    verbose                  = False,                             # set True to see per-spec load lines
)

print(f'group: {len(group)} sessions, label = {group.label!r}')
print()
print('members each session provides:')
for name in group[0].names:
    print(f'  - {name}')

Stack the Nosepoke activations across every session into one combined `xr.DataArray`. The result still has its Nosepoke shape (`Time x peripherals`) and now carries a `'label'` coordinate naming the source session for every sample.

In [ ]:
nosepokes = combine_sessions(
    group       = group,
    dim         = 'Time',                       # default; stack along the time axis
    type_name   = 'nosepoke:Activations',       # combine just this one member
    label_coord = 'label',                      # default; provenance coord name
    on          = 'intersection',               # default; keep members present in every session
)

print('combined nosepokes:', dict(nosepokes.sizes))
print('label coord values (first 5):', list(nosepokes['label'].values[:5]))

**The all-structures, all-sessions object.** Omit `type_name` (or pass `type_name=None`) to combine every member at once. The result is a single `Session` whose `data` dict holds every structure from the catalog, each stacked across every session in the group on the `dim` axis. This is the canonical 'everything together' object that section 2.3 and section 3 build on.

In [ ]:
# Combine every member at once. The result is a Session whose data dict
# contains every structure in the catalog, each stacked across every
# session in the group on the Time axis (or whichever `dim` you pass).
grouped_sessions = combine_sessions(
    group       = group,
    dim         = 'Time',
    type_name   = None,                         # None -> combine every member; returns a Session
    label_coord = 'label',
    on          = 'intersection',
)

print(f'grouped_sessions: {len(grouped_sessions.names)} members, {len(group)} sessions stacked into each.\n')

# Build a summary DataFrame, one row per member, columns = total samples and
# number of distinct sessions tagged. Last expression -> Jupyter renders it.
summary_rows = []
for name in grouped_sessions.names:
    obj = grouped_sessions.data[name]
    if hasattr(obj, 'sizes'):                          # xarray
        total = obj.sizes.get('Time', 0)
        labels = obj['label'].values if 'label' in obj.coords else []
        dtype = type(obj).__name__
    else:                                              # pandas DataFrame
        total = len(obj)
        labels = obj['label'].values if 'label' in obj.columns else []
        dtype = 'DataFrame'
    summary_rows.append({
        'member':           name,
        'type':             dtype,
        'total_rows':       total,
        'sessions_tagged':  len(set(labels)),
    })

summary = pd.DataFrame(summary_rows).set_index('member')
summary

### 2.3. Pulling individual structures out of `grouped_sessions`

`grouped_sessions.data` is a plain `dict[str, xr.DataArray | xr.Dataset | pd.DataFrame]` keyed by member name. Every member has been stacked across sessions on its `dim` axis (default `'Time'`) and tagged with the source-session label (a `'label'` coord on xarray members; a `'label'` column on DataFrame members). To pull one structure out for analysis, just index `grouped_sessions.data['<member_name>']`.

The cells below pull one of each shape that the default catalog produces: an `xr.DataArray` (the Nosepoke activations), a `pd.DataFrame` (the events table), and an `xr.Dataset` (a SoundCard register).

In [ ]:
# Inspect what is in grouped_sessions.data: name, type, and the dim it spans.
print('members in grouped_sessions.data:')
for name in grouped_sessions.names:
    obj = grouped_sessions.data[name]
    print(f'  {name:40s}  ({type(obj).__name__})')
print()

# Pull the Nosepoke activations: an xr.DataArray (Time x peripherals),
# carrying a 'label' coord on Time that names the source session for
# each sample.
activations = grouped_sessions.data['nosepoke:Activations']
print('activations sizes        :', dict(activations.sizes))
print('activations dims         :', activations.dims)
print('distinct session labels  :', len(set(activations['label'].values)),
      f'(expected {len(group)})')
print()

# Concrete proof: count samples contributed by each session along Time.
print('samples contributed per session (first 8 sessions):')
print(pd.Series(activations['label'].values).value_counts().sort_index().head(8))

In [ ]:
# Pull the ExperimentEvents table from grouped_sessions. It is one
# pd.DataFrame stacked from every session's events log, with a 'label'
# column tagging each row's source session.
events = grouped_sessions.data['events']

# Per-session row counts. Every session in the group should be represented.
per_session = events.groupby('label').size().rename('n_rows').reset_index()
print(f'events: {len(events):,} rows across {len(per_session)} sessions\n')
print('per-session row counts:')
display(per_session)

# The full events DataFrame. The 'label' column flips as the table moves
# from session to session: scroll through to see the boundaries.
events

Per-session view: each session's events as its own DataFrame. This is the same data the combined `events` table above carries, just sourced from `group[i].data['events']` (the SessionGroup keeps the per-session split alongside the combined view). First three sessions shown below.

In [ ]:
for session in list(group)[:3]:
    label = session.metadata.get('label', '?')
    df = session.data['events']
    print(f'\n== session {label}  ({len(df):,} rows) ==')
    display(df.head(5))

In [ ]:
# Pull a SoundCard register: an xr.Dataset stacked across sessions on the
# Time dimension. Convert to a DataFrame for display so the per-row 'label'
# coord lands as a column.
playsound = grouped_sessions.data['soundcard:PlaySoundFreq']
print(f'PlaySoundFreq sizes: {dict(playsound.sizes)}  data_vars: {list(playsound.data_vars)}\n')
playsound.to_dataframe().reset_index()

## 3. Querying and Interacting with the Combined Data

Throughout section 3 we work on `nosepokes = grouped_sessions.data['nosepoke:Activations']`. It is the same object that the standalone `combine_sessions(..., type_name='nosepoke:Activations')` call in section 2.2 produced; sourcing it from `grouped_sessions` keeps the narrative tied to the canonical all-in-one Session.

`nosepokes` is still an `xr.DataArray`, so all of xarray's selection machinery works on it. data-conduit adds two helpers, `add_label_column` and `drop_columns`, for tailoring the combined object when you want a derived grouping or want to drop coords in a final table. The `.ulookup` accessor (from `data_conduit.virtualarrays`) lets you query along the virtual coordinates attached during the multi-device combine (`device`, `register`, `localID`).

Three things in this section:

1. Querying with `.ulookup`: filter the combined Nosepoke by virtual coordinate.
2. Adding a derived label with `add_label_column`: tag every row with the day it came from.
3. Dropping coords with `drop_columns`: remove the per-row virtual coords for a tidy table.

### 3.1. Querying with `.ulookup`

The Nosepoke MultiDevice (built by the `Nosepoke` preset in section 0.4) attaches three virtual coordinates to every sample of the `peripherals` dimension: `device`, `register`, `localID`. After `combine_sessions`, these still survive, so you can ask things like 'give me only Behavior0' or 'only the activation inputs (`register='32'`)' without touching the underlying data.

In [ ]:
# Source the activations from grouped_sessions so section 3 stays tied to the canonical object.
nosepokes = grouped_sessions.data['nosepoke:Activations']

# Just the activations on Behavior0 (3 nosepokes per board).
board0 = nosepokes.ulookup.select(device='Behavior0')
print('device=Behavior0     :', dict(board0.sizes))

# Just one specific port (NP_0 == Behavior0/DIPort0).
np0 = nosepokes.ulookup.select(device='Behavior0', localID='DIPort0')
print('Behavior0 / DIPort0  :', dict(np0.sizes))

# All sessions still tagged on the Time dim; show the first few labels.
print('first 5 session labels on the filtered slice:', list(np0['label'].values[:5]))

### 3.2. Adding a derived label with `add_label_column`

`add_label_column(obj, name, value_or_fn, dim='Time')` attaches a new coordinate (for an xarray object) or column (for a DataFrame) along the alignment axis. The third argument is either a scalar (broadcast to every element) or a callable `fn(obj) -> array` that returns one value per element. Use it to tag rows with a derived grouping (a 'block', a 'condition', or, as here, the `day` each row came from).

In [ ]:
# Map each session label back to the 'day' metadata captured by select_sessions.
label_to_day = {name: entry['day'] for name, entry in sessions.items()}

# Attach a 'day' coord that runs along the Time axis. The callable receives the
# combined object and must return one value per Time sample.
with_day = add_label_column(
    obj         = nosepokes,
    name        = 'day',
    value_or_fn = lambda da: np.array([label_to_day[label] for label in da['label'].values]),
    dim         = 'Time',
)

print('day coord now present:', 'day' in with_day.coords)
print()
print('samples per day:')
print(pd.Series(with_day['day'].values).value_counts().sort_index())

### 3.3. Dropping coords with `drop_columns`

The mirror of `add_label_column`. Useful when shaping a final table for export or plotting: drop the bookkeeping coords (`device`, `register`, `localID`) once you no longer need them.

In [ ]:
# Drop the virtual coords (kept on the peripherals dim), then convert to a
# long DataFrame for downstream use. NaN rows are dropped: the Activations
# array is sparse, with NaN where a port did not fire on that sample.
tidied_df = (
    drop_columns(with_day, ['device', 'register', 'localID'])
      .to_dataframe()
      .dropna(subset=['Activations_data'])
      .reset_index()
)
print(f'tidy table: {len(tidied_df):,} rows, columns = {list(tidied_df.columns)}')
tidied_df.head(10)

## 4. Cross-Clock Data (Neuropixels, DLC). Sketch only.

Every structure in the catalog so far has shared the same Bonsai / HARP clock. Data recorded on a different clock (Neuropixels, DeepLabCut pose) needs a TTL conversion onto the reference clock BEFORE it can be merged with the rest of the bundle. The sessiongroups package wires this in via the `sync` field on a `DataStructureSpec` and the `attach_cross_clock` function.

The mechanism in one paragraph:

1. The cross-clock spec's `sync` dict carries the configuration the TTL sync utilities need, including a `'reference_pulses'` entry that is either a pulse-table `DataFrame` or a `callable(bundle) -> DataFrame` that builds it from the already-loaded same-clock bundle.
2. The spec's `reader` returns `{'pulse_table', 'data', 'time_column'}`: a pulse table on the structure's OWN clock, the data to merge in, and the name of the data's time axis.
3. `attach_cross_clock(session, catalog)` then fits a TTL conversion (via `data_conduit.sync.ttl`), applies it to the data's time axis, shifts by `t0` if the session has been normalised, and merges the converted members into a new Session. The fitted models are returned alongside so the caller can cache them.

These readers are CALLER-SUPPLIED: the package never imports `movement`, SpikeInterface, or any Neuropixels library. The sketch below shows the shape; it does not run on the example data (which has no Neuropixels recordings).

In [ ]:
# NOT RUN: requires Neuropixels data not shipped with the example.
#
# from data_conduit.sync.ttl import build_pulse_table, extract_ttl_segments
#
#
# def npx_reader(session_path):
#     '''Read raw NPX sync channel + spike times for one session.
#
#     Returns a dict in the shape attach_cross_clock expects.
#     '''
#     # 1) Read the raw TTL sync channel sampled on the NPX clock.
#     raw = np.load(session_path / 'npx' / 'sync_channel.npy')
#     times = np.arange(len(raw)) / 30000.0          # 30 kHz NPX clock
#
#     # 2) Extract pulses on the NPX clock (a DataFrame with 'Start', 'End', 'Duration').
#     npx_pulses, _ = extract_ttl_segments(times=times, values=raw)
#
#     # 3) The data to merge: spike times, with a column carrying the timestamp.
#     spikes = pd.read_parquet(session_path / 'npx' / 'spike_times.parquet')
#
#     return {
#         'pulse_table': npx_pulses,
#         'data':        spikes,
#         'time_column': 'spike_time',
#     }
#
#
# # The reference pulses live in the events stream that load_session already loaded.
# # Build them once per session from the bundle, via a callable on the sync dict.
# def bonsai_reference_pulses(bundle):
#     events = bundle['events']
#     return build_pulse_table(
#         rise_times = events.query('Event == "TTL Rising"').index.to_numpy(),
#         fall_times = events.query('Event == "TTL Falling"').index.to_numpy(),
#     )
#
#
# catalog.add(DataStructureSpec(
#     name     = 'npx',
#     reader   = npx_reader,
#     required = False,                      # skip sessions without NPX data
#     sync     = {                            # presence of this dict marks the spec cross-clock
#         'reference_pulses': bonsai_reference_pulses,
#         'use':              'start',        # forwarded to data_conduit.sync.ttl
#     },
# ))
#
#
# # On the per-session pipeline, attach_cross_clock now does the work:
# session = load_session(one_path, catalog)
# session = normalise_to_zero(session)
# session, ttl_models = attach_cross_clock(session, catalog)
#
# # The session now carries 'npx:spike_times' (or whatever the reader returned),
# # converted onto the bonsai clock and t0-shifted. ttl_models['npx'] holds the
# # fitted conversion model in case you want to cache or reuse it.